# Robustness of the reported conclusions to marker-alignment quality

A small number of recordings carry large clock-fit residuals, indicating that their stimulus markers are misaligned with the EEG (`notebooks/timing_diagnostic.ipynb`). This notebook asks the only question that bears on the manuscript: **do the reported conclusions depend on those recordings?**

The question is answered by **exclusion rather than correction**. Re-aligning affected recordings would mean adopting a different clock estimator across the whole dataset and re-deriving every reported value. Excluding them and showing the conclusions hold is the conventional and weaker move, and it is sufficient. No reported value is altered by anything here.

## Post-hoc status

The residual thresholds used below were chosen **after** the residual distribution was known. They are not a pre-specified rule and are not presented as one. Every analysis is therefore reported at two cuts — the three recordings with the largest residuals, and all eleven above a looser flag — so that a reader can see whether a conclusion depends on where the line is drawn.

## Prerequisite

`data/derived/timing-diagnostic/residual_scan.csv` must exist. If it does not, run section 2 of `notebooks/timing_diagnostic.ipynb` first.

In [ ]:
import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
_here = Path('.').resolve()
repo_root = next((p for p in [_here, _here.parent, _here.parent.parent]
                  if (p / 'config.yaml').exists()), _here)
os.chdir(repo_root); sys.path.insert(0, str(repo_root))

%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from analysis import robustness_checks as rc

print('repo root:', repo_root)
print(f'severe cut: {rc.CUT_SEVERE_MS:.0f} ms | '
      f'flagged cut: {rc.CUT_FLAGGED_MS:.0f} ms  (both post-hoc)')
print('subjects above severe cut :', rc.subjects_above_residual(rc.CUT_SEVERE_MS))
print('subjects above flagged cut:', rc.subjects_above_residual(rc.CUT_FLAGGED_MS))

## 1. Primary analysis: balanced accuracy

Friedman omnibus followed by Holm-corrected one-tailed Wilcoxon against held-out control, matching the procedure in `analysis/classifier.py`. Exclusion operates at subject level, since the design is within-subject and the group tests use list-wise deletion.

In [ ]:
prim = rc.exclusion_sensitivity('classifier-v3')

rows = []
for label, v in prim['variants'].items():
    row = {'variant': label, 'N': v['n'], 'control': round(v['reference_mean'], 2),
           'friedman_p': f"{v['friedman']['p']:.2g}"}
    for cond, c in v['contrasts'].items():
        row[f'{cond}_mean'] = round(c['mean'], 2)
        row[f'{cond}_dz'] = round(c['dz'], 2)
        row[f'{cond}_p'] = f"{c['p_holm']:.2g}"
    rows.append(row)
display(pd.DataFrame(rows).set_index('variant').T)

for label, v in prim['variants'].items():
    ch = v['contrasts']['chewing']
    print(f"{label:26s} N={v['n']:>2}  chewing d_z={ch['dz']:+.2f} "
          f"(p_holm={ch['p_holm']:.2g}, significant={ch['significant_holm']})")

The chewing contrast is unchanged under either exclusion. The EMI and acoustic contrasts, already non-significant, move toward and slightly past zero: the between-subject spread that distinguished EMI from the other conditions is contributed by the excluded recordings rather than by the manipulation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
labels = list(prim['variants'])
conds = ['chewing', 'emi', 'acoustic']
w = 0.26
for i, cond in enumerate(conds):
    vals = [prim['variants'][l]['contrasts'][cond]['dz'] for l in labels]
    ax.bar(np.arange(len(labels)) + (i - 1) * w, vals, w, label=cond)
ax.axhline(0, color='#444', lw=.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([l.replace('drop_residual_gt_', '> ').replace('ms', ' ms')
                    for l in labels], fontsize=8)
ax.set_ylabel("Cohen's $d_z$ vs control")
ax.set_title('Primary contrasts under alignment-based exclusion')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 2. Secondary analysis: N170 amplitude

In [ ]:
sec = rc.n170_exclusion_check()
for label, v in sec['variants'].items():
    ch = v['contrasts']['chewing']
    print(f"{label:26s} N={v['n']:>2}  chewing d={ch['dz']:+.2f} "
          f"(mean {ch['mean']:+.3f} uV, p_holm={ch['p_holm']:.2g})")
    for cond in ('emi', 'acoustic'):
        c = v['contrasts'][cond]
        print(f"{'':26s}      {cond:8s} d={c['dz']:+.2f} p_holm={c['p_holm']:.2g}")

### 2b. The chewing / rejection-rate convergence

Two definitions of the dependent measure are reported because they differ by roughly 0.03. The manuscript value corresponds to the **raw chewing amplitude**. Both appear here so that the reported figure is not accidentally replaced by the other.

In [ ]:
conv = rc.n170_convergence_check()
rows = []
for label, v in conv.items():
    rows.append({'variant': label, 'n': v['n'],
                 'raw amplitude r': round(v['raw_amplitude']['pearson_r'], 3),
                 'shift r': round(v['shift_from_control']['pearson_r'], 3),
                 'raw Spearman': round(v['raw_amplitude']['spearman_rho'], 3)})
display(pd.DataFrame(rows).set_index('variant'))

## 3. Shape of the EMI accuracy distribution

The manuscript characterises the EMI distribution as having a heavy lower tail. Recomputing its shape statistics without the alignment-affected subjects shows how much of that shape those recordings account for.

In [ ]:
emi = rc.emi_distribution_check()
rows = []
for label, v in emi.items():
    r = {'variant': label, 'n': v['n'], 'mean': round(v['mean'], 2),
         'SD': round(v['sd'], 2), 'skew': round(v['skewness'], 2),
         'excess kurtosis': round(v['excess_kurtosis'], 2),
         'bimodality coef': round(v['bimodality_coefficient'], 3),
         'exceeds 0.555': v['exceeds_bimodality_threshold']}
    if v['hartigan_dip']:
        r['dip p'] = round(v['hartigan_dip']['p_bootstrap'], 3)
    rows.append(r)
display(pd.DataFrame(rows).set_index('variant'))

## 4. Below-chance ranking as an independent detector

Additive noise degrades AUC toward 0.5 and cannot pass it, so a classifier ranking targets *below* non-targets indicates a systematic error rather than a noisy recording. This makes AUC an independent detector of misalignment, requiring no timing information at all.

In [ ]:
bc = rc.below_chance_scan()
if not bc.get('available'):
    print(bc['note'])
else:
    print(f"recordings scanned: {bc['n_recordings']}")
    for key in ('below_0.50', 'below_0.60'):
        print(f"\nAUC {key.replace('below_', '< ')}:  n = {bc[key]['n']}")
        if bc[key]['n']:
            display(pd.DataFrame(bc[key]['recordings']).round(3))

## 5. The registered exclusion rule

The registered rule excluded any subject exceeding 20% epoch rejection on *any* condition; the analysis actually run narrowed this to the control condition alone (`DEVIATIONS.md`, 2026-06-07). The manuscript reports this comparison for the published pipeline; it is recomputed here for the blockwise pipeline as well.

In [ ]:
gate = rc.registered_gate_recompute()
for pipe, v in gate['pipelines'].items():
    print(f"{pipe:26s} N={v['n']}  control={v['control_mean']:.2f}  "
          f"chewing={v['chewing_mean']:.2f}  d_z={v['dz']:+.2f}  "
          f"p={v['p_one_tailed_uncorrected']:.2g} (uncorrected, one-tailed)")

## 6. Write the summary

`robustness_summary.json` is the canonical record of these figures. Values quoted elsewhere should be read from it rather than transcribed.

In [ ]:
results = rc.run_all()
path = rc.save_summary(results)
print('saved ->', path)
print(f'{path.stat().st_size / 1024:.1f} KB\n')

for label, v in results['primary_accuracy']['variants'].items():
    ch = v['contrasts']['chewing']
    print(f"  {label:26s} N={v['n']:>2}  chewing d_z={ch['dz']:+.2f}  "
          f"p_holm={ch['p_holm']:.2g}")